# 🦀 Day 6: Serving Static Files & HTML Templates

---

## Overview

Today we cover:
- Serving static files (CSS, JS, images) with `tower-http::ServeDir`
- HTML template engines: **Askama** (compile-time, type-safe) vs **Tera** (runtime)
- Template syntax: variables, loops, conditionals, inheritance
- Rendering templates in Axum handlers

In [ ]:
:dep axum = "0.7"
:dep tokio = { version = "1", features = ["full"] }
:dep serde = { version = "1", features = ["derive"] }
:dep askama = "0.12"

// Dependencies loaded. Note: tower-http is needed for ServeDir.

## Serving Static Files with tower-http

Use `tower_http::services::ServeDir` to serve files from a directory. Mount it as a route or as a fallback.

In [ ]:
// === Cargo project code — EvCxR cannot run a server ===
//
// :dep tower-http = { version = "0.5", features = ["fs"] }
//
// use axum::{Router, routing::get};
// use tower_http::services::ServeDir;
//
// // Serve files from ./static at /static/*
// let app = Router::new()
//     .nest_service("/static", ServeDir::new("static"));
//
// // Or serve at root: any unmatched path tries static/
// let app = Router::new()
//     .fallback_service(ServeDir::new("static"));

println!("ServeDir::new("static") serves files from the static/ directory.");

## HTML Template Engines Overview

| Engine | Type | Pros | Cons |
|--------|------|------|------|
| **Askama** | Compile-time | Type-safe, fast, no runtime errors | Requires rebuild for template changes |
| **Tera** | Runtime | Hot-reload, Jinja2-like | Slower, runtime template errors |

We'll focus on **Askama** — it compiles templates into Rust code, so you get compile-time checks.

## Askama: Type-Safe, Compile-Time Templates

Askama uses Jinja2-like syntax. Create a struct that implements `Template` and define the template as a string or in a `.html` file.

In [ ]:
use askama::Template;

// Inline template: use source for notebooks (path="..." requires a file in Cargo)
#[derive(Template)]
#[template(source = "Hello {{ name }}! Count: {{ count }}", ext = "txt")]
struct HelloTemplate {
    name: String,
    count: u32,
}

let t = HelloTemplate { name: "Rust".into(), count: 42 };
println!("{}", t.render().unwrap());

## Template Syntax: Variables, Loops, Conditionals

| Syntax | Example |
|--------|----------|
| Variable | `{{ name }}` |
| Loop | `{% for item in items %}...{% endfor %}` |
| Conditional | `{% if condition %}...{% else %}...{% endif %}` |
| Inheritance | `{% extends "base.html" %}` + `{% block content %}...{% endblock %}` |

In [ ]:
use askama::Template;

#[derive(Template)]
#[template(source = r#"
Items:
{% for item in items %}
  - {{ item }}
{% endfor %}
{% if items.is_empty() %}
  (no items)
{% endif %}
"#, ext = "txt")]
struct ListTemplate {
    items: Vec<String>,
}

let t = ListTemplate {
    items: vec!["Rust".into(), "Axum".into(), "Askama".into()],
};
println!("{}", t.render().unwrap());

## File Structure for Askama (Cargo Project)

In a Cargo project, put templates in `templates/` and reference by path:

```
templates/
  base.html      # Base layout with {% block content %}
  index.html     # {% extends "base.html" %}
  list.html      # List page template
```

Add to `Cargo.toml`:
```toml
[package]
build = "build.rs"

[build-dependencies]
askama = "0.12"
```

`build.rs` can be minimal — Askama's derive finds templates by default in `templates/`.

## Template Inheritance: base.html

**templates/base.html:**
```html
<!DOCTYPE html>
<html>
<head><title>{% block title %}My App{% endblock %}</title></head>
<body>
  {% block content %}{% endblock %}
</body>
</html>
```

**templates/list.html:**
```html
{% extends "base.html" %}
{% block title %}Items{% endblock %}
{% block content %}
<ul>
{% for item in items %}
  <li>{{ item }}</li>
{% endfor %}
</ul>
{% endblock %}
```

## Rendering Templates in Axum Handlers

Return the template's HTML via `Html(template.render().unwrap())`. Use `axum::response::Html`.

In [ ]:
use axum::{Router, routing::get, response::Html};
use askama::Template;

#[derive(Template)]
#[template(source = "<h1>Hello {{ name }}</h1>", ext = "html")]
struct PageTemplate { name: String }

async fn index() -> Html<String> {
    let t = PageTemplate { name: "World".into() };
    Html(t.render().unwrap())
}

let app = Router::new().route("/", get(index));

// Verify it renders
let t = PageTemplate { name: "Rust".into() };
println!("Rendered: {}", t.render().unwrap());

## Passing Data to Templates

The template struct holds all data. Use `serde::Serialize` for nested structs if needed. Askama supports:
- Primitives, String, Option
- Vec, HashMap
- Nested structs (with Serialize or Template)

In [ ]:
use askama::Template;

// Nested struct: the inner struct's fields are accessible as user.name, user.id
#[derive(Template)]
#[template(source = "User: {{ name }}, ID: {{ id }}, Tags: {% for t in tags %}{{ t }}{% if not loop.last %}, {% endif %}{% endfor %}", ext = "txt")]
struct UserInfo {
    name: String,
    id: u32,
    tags: Vec<String>,
}

#[derive(Template)]
#[template(source = "Page: {{ user.name }} (id {{ user.id }})", ext = "txt")]
struct UserPage {
    user: UserInfo,
}

let user = UserInfo { name: "Alice".into(), id: 1, tags: vec!["a".into(), "b".into()] };
let page = UserPage { user };
println!("{}", page.render().unwrap());

## 🏋️ Exercise: List Page Template

Create a template for a list page with items. Each item has `id`, `title`, and `url`. Use a loop to render a list. Add a conditional to show "No items yet" when the list is empty.

In [ ]:
// YOUR CODE HERE
//
// struct Item { id: u32, title: String, url: String }
// struct ListPage { items: Vec<Item> }
// Template: loop over items, show id, title, link (url)
// If empty, show "No items yet"
//
// use askama::Template and #[template(source = "...", ext = "html")]

## 🎯 Key Takeaways

1. **tower-http ServeDir** serves static files from a directory
2. **Askama** = compile-time, type-safe templates; **Tera** = runtime, Jinja2-like
3. Template syntax: `{{ var }}`, `{% for %}`, `{% if %}`, `{% extends %}`, `{% block %}`
4. Return `Html(template.render().unwrap())` from Axum handlers
5. Put templates in `templates/` for Cargo projects; use `path = "name.html"`
6. Template inheritance: base.html with blocks, child templates extend and override

---

**Tomorrow:** Mini-project — REST API bookmark manager! 🔖